# depth_bench on Google Colab

Runs the full benchmark (`depth_bench/run_all.py`): zero-shot evaluation, then training and evaluation of all models on all folds.

**Before the first run:** upload the folder made by `python -m depth_bench.pack_for_colab` (`MODEST_colab`: 9 scene tars, `manifest.csv`, `dfocus_aperture_pairs.csv`, 36 GB) to Google Drive, under `MyDrive/MODEST_colab`.

**After a disconnect:** run all cells again. Finished steps are skipped and stopped training resumes from `last.pt`. All results are written to Google Drive (`MyDrive/MODEST_runs`).

Keep the browser tab open: with Colab Pro, the session stops when the tab is closed.

## 1. Settings

In [ ]:
DRIVE_DATA = "/content/drive/MyDrive/MODEST_colab"   # uploaded tars + manifest + focus file
DRIVE_RUNS = "/content/drive/MyDrive/MODEST_runs"    # all outputs (checkpoints, logs, results)
REPO = "https://github.com/sleekEagle/RealCameraIndependent"
FOLDS = ["S1", "S2", "S3", "S4", "S5"]
MODELS = ["da2", "unidepth", "metric3d", "da3", "camind"]

## 2. GPU and Google Drive

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
from google.colab import drive
drive.mount("/content/drive")

## 3. Code: this repository and the four model repositories (fixed versions)

In [ ]:
%%bash -s "$REPO"
set -e
cd /content
if [ -d RealCameraIndependent ]; then git -C RealCameraIndependent pull -q; else git clone -q "$1" RealCameraIndependent; fi
mkdir -p code_ext && cd code_ext
clone() {  # name url commit
  if [ ! -d "$1" ]; then git clone -q "$2" "$1"; fi
  git -C "$1" checkout -q "$3"
}
clone Depth-Anything-V2 https://github.com/DepthAnything/Depth-Anything-V2   a561b849ebae10a6f5ef49e26c83cbbcd36c71bf
clone UniDepth          https://github.com/lpiccinelli-eth/UniDepth          8d8cfe4c7ee15297099983607febf0d4f32eb3d6
clone Metric3D          https://github.com/YvanYin/Metric3D                  eb5b6fac0dc155e4e52f576e304fbf11655ff339
clone Depth-Anything-3  https://github.com/ByteDance-Seed/Depth-Anything-3   3d835ec1a5802d64a8b8b15f817a1ab54809bfe4
echo code ready

## 4. Packages

Colab's own PyTorch is kept. UniDepth and Depth Anything 3 are installed with `--no-deps`: their pinned requirements conflict (numpy), and both work with Colab's numpy.

In [ ]:
%%bash
set -e
pip install -q opencv-python-headless tifffile imagecodecs pillow pandas scipy tqdm huggingface_hub safetensors                timm einops addict omegaconf wandb h5py tabulate termcolor imageio mmengine
pip install -q --no-deps -e /content/code_ext/UniDepth -e /content/code_ext/Depth-Anything-3
python -c "import torch, numpy; print('torch', torch.__version__, 'numpy', numpy.__version__, 'cuda', torch.cuda.is_available())"

## 5. Data: extract the tars from Drive to the local disk

About 36 GB. Reading training data from Drive directly would be far too slow. Skipped if already extracted in this session.

In [ ]:
%%bash -s "$DRIVE_DATA"
set -e
mkdir -p /content/MODEST_depth
for t in "$1"/Scene*.tar; do
  done_flag=/content/MODEST_depth/.$(basename "$t").done
  if [ ! -f "$done_flag" ]; then echo "extracting $(basename "$t")"; tar -xf "$t" -C /content/MODEST_depth && touch "$done_flag"; fi
done
cp "$1"/manifest.csv /content/MODEST_depth/
cp "$1"/dfocus_aperture_pairs.csv /content/
df -h /content | tail -1

## 6. Paths for depth_bench

In [ ]:
import os
os.environ.update({
    "MODEST_DEPTH_ROOT": "/content/MODEST_depth",
    "MODEST_FOCUS_CSV": "/content/dfocus_aperture_pairs.csv",
    "DEPTH_BENCH_CODE_EXT": "/content/code_ext",
    "DEPTH_BENCH_RUNS": DRIVE_RUNS,
    "HF_HOME": "/content/hf_cache",
    "TORCH_HOME": "/content/torch_cache",
})
os.makedirs(DRIVE_RUNS, exist_ok=True)
%cd /content/RealCameraIndependent
!python -m depth_bench.run_all --dry_run --folds {" ".join(FOLDS)} --models {" ".join(MODELS)}

## 7. Run

Prints key lines only; the full output of every step is in `MODEST_runs/logs/`. Re-run this cell (after cells 1-6) to continue after a disconnect.

In [ ]:
!python -m depth_bench.run_all --folds {" ".join(FOLDS)} --models {" ".join(MODELS)}

## 8. Results so far

In [ ]:
import glob, pandas as pd
rows = []
for f in sorted(glob.glob(f"{DRIVE_RUNS}/eval/*/summary.csv")):
    s = pd.read_csv(f)
    s.insert(0, "run", f.split("/")[-2])
    rows.append(s[s.group.str.startswith("S")])
pd.concat(rows)[["run", "group", "value", "n_crops", "absrel", "absrel_med", "delta1", "si_absrel", "scale", "bf1"]] if rows else "no results yet"